# STEW mental workload — end-to-end reproduction

Aligned with **`PRD.md`** and **`README.md`**: one notebook drives `src/stew_mwl/`.

- Load experiment settings from **`configs/*.yaml`** (paths resolved to this repo root).
- Build a manifest from **`data/STEW/`** (14-ch text files + ratings file).
- Classes **BL, LW, MW, HW** (baseline `lo` file → BL; task ratings 1–3 / 4–6 / 7–9 → LW / MW / HW).
- **LOSO** training for the **VAE + CBAM + BiLSTM** proposed model; **PSD-SVM**, **CNN**, **BLSTM–LSTM** baselines; **ablations**; optional **sensitivity** grids (`run_sensitivity` in YAML).
- Export PRD-listed CSVs under **`outputs/csv/`**, models under **`outputs/models/`**, figures under **`outputs/figures/`**.
- **Grad-CAM** regional summaries (`gradcam_region_summary.csv`, `gradcam_sample_scores.csv`) using anterior/posterior grid proxies (cap lacks Fp1/Fz/Pz).

Switch YAML file below: `default.yaml` = fast smoke; `full_reproduction.yaml` = full 48-subject-style run (long).

### Project paths

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)
print("SRC:", SRC)

### Imports

In [ ]:
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display

print("Python:", sys.version.split()[0], "| TensorFlow:", tf.__version__)

from stew_mwl.config import CLASS_NAMES
from stew_mwl.yaml_loader import load_config_from_yaml
from stew_mwl.data import set_global_determinism, build_subject_manifest, validate_stew_dataset
from stew_mwl.train import (
    run_loso_training,
    run_baseline_models,
    run_ablation_variants,
    run_sensitivity_grids,
)
from stew_mwl.eval import aggregate_fold_metrics, paired_ttest_vs_full
from stew_mwl.export import (
    export_dataset_manifest,
    export_segmentation_summary,
    export_fold_metrics_and_predictions,
    export_classification_report_and_cm,
    export_baseline_comparison_summary,
    export_ablation_summary,
    export_statistical_tests,
    export_experiment_registry,
    export_gradcam_summaries,
)
from stew_mwl import plotting
from stew_mwl.gradcam import run_gradcam_export_for_checkpoint

### Configuration (YAML)

All major settings come from **`configs/default.yaml`** or **`configs/full_reproduction.yaml`**. `load_config_from_yaml(..., project_root=ROOT)` anchors `data_root` and `output_root` to the repository root so outputs do not land under `notebooks/`.

In [ ]:
CONFIG_FILE = ROOT / "configs" / "default.yaml"
# CONFIG_FILE = ROOT / "configs" / "full_reproduction.yaml"

cfg = load_config_from_yaml(CONFIG_FILE, project_root=ROOT)
set_global_determinism(cfg.seed)
cfg.ensure_dirs()

print(cfg)
print("Classes:", CLASS_NAMES)
print("data_root:", cfg.data_root)
print("CSV dir:", cfg.csv_dir)
print("Models dir:", cfg.models_dir)

## Dataset

Place STEW under `data/STEW/` (see repository `README.md`). The pipeline expects `subXX_lo.txt`, `subXX_hi.txt`, and a ratings file whose name contains "rating".

In [ ]:
manifest = build_subject_manifest(cfg)
issues = validate_stew_dataset(cfg, manifest)
if issues:
    print("Dataset validation warnings:")
    for msg in issues:
        print(" -", msg)
else:
    print("Dataset validation: OK")

display(manifest.head())
manifest.shape, manifest["task_label"].value_counts()

### Export manifest and segmentation counts

In [ ]:
export_dataset_manifest(manifest, cfg)
export_segmentation_summary(manifest, cfg)
print("Wrote:", cfg.csv_dir / "dataset_manifest.csv")
print("Wrote:", cfg.csv_dir / "segmentation_summary.csv")

## Proposed model — LOSO training

Writes per-fold metrics, per-trial predictions, optional `vae_fold_losses.csv`, and Keras checkpoints under `outputs/models/`.

In [ ]:
full_df, loso_splits, predictions = run_loso_training(cfg, manifest, model_name="proposed")
_, full_summary = aggregate_fold_metrics(full_df.to_dict("records"))
full_summary

In [ ]:
export_fold_metrics_and_predictions(full_df, predictions, cfg)
export_classification_report_and_cm(predictions, cfg)
print("Wrote fold_metrics_all.csv, predictions_all.csv, classification_report_proposed.csv, confusion_matrix_proposed.csv")

## Baselines (same LOSO splits)

In [ ]:
baseline_results = run_baseline_models(cfg, manifest, loso_splits=loso_splits)
export_baseline_comparison_summary(baseline_results, cfg)
for name, df in baseline_results.items():
    print(name, "folds:", len(df))
    if len(df):
        print(df[["subject", "accuracy", "macro_f1"]].head())

## Ablations

In [ ]:
ablation_results = run_ablation_variants(
    cfg, manifest, loso_splits=loso_splits, full_fold_metrics_df=full_df
)
export_ablation_summary(ablation_results, full_df, cfg)
export_statistical_tests(full_df, baseline_results, ablation_results, cfg)
export_experiment_registry(cfg, notes="notebook stew_mwl_reproduction")
list(ablation_results.keys())

### Example: paired t-test vs proposed (accuracy)

In [ ]:
if "blstm_lstm" in baseline_results and len(baseline_results["blstm_lstm"]):
    p = paired_ttest_vs_full(full_df, baseline_results["blstm_lstm"], "accuracy")
    print("paired t-test p (proposed vs blstm_lstm, accuracy):", p)

## Sensitivity grids

PRD Stage K: latent size {64,128,256}, CBAM configs, window {5,10,15}s, sequence steps {5,10,15}. Runs when **`cfg.run_sensitivity`** is true in YAML (e.g. `full_reproduction.yaml`). Very expensive for 48 subjects; `quick_mode` uses a subject subset inside `run_sensitivity_grids`.

In [ ]:
if cfg.run_sensitivity:
    sens = run_sensitivity_grids(cfg, manifest)
    print("Sensitivity keys:", list(sens.keys()))
else:
    print("Skipped (cfg.run_sensitivity is False). Set true in YAML to run.")

## Figures

In [ ]:
from sklearn.metrics import confusion_matrix

pred_df = pd.DataFrame(predictions)
if len(pred_df):
    yt = pred_df["y_true"].values
    yp = pred_df["y_pred"].values
    cm = confusion_matrix(yt, yp, labels=list(range(len(CLASS_NAMES))))
    plotting.plot_confusion_matrix(cm, cfg)
plotting.plot_vae_loss_curves(cfg.csv_dir / "vae_fold_losses.csv", cfg)
plotting.plot_baseline_bar(cfg.csv_dir / "baseline_comparison_summary.csv", cfg)
plotting.plot_ablation_bar(cfg.csv_dir / "ablation_summary.csv", cfg)
print("Figures under:", cfg.figures_dir)

## Grad-CAM (PRD Stage N)

Exports **`gradcam_region_summary.csv`** and **`gradcam_sample_scores.csv`**: mean activation in **anterior vs posterior halves** of the topomap grid (proxies for frontal vs parietal; STEW montage does not include Fp1/Fz/Pz). Uses the first saved proposed checkpoint if present.

In [ ]:
ckpts = sorted(cfg.models_dir.glob("fold_*_proposed.keras"))
if ckpts:
    reg_df, samp_df = run_gradcam_export_for_checkpoint(
        cfg, manifest, ckpts[0], max_samples=16, save_example_figure=True
    )
    export_gradcam_summaries(reg_df, samp_df, cfg)
    plotting.plot_gradcam_outputs_on_disk(cfg)
    print("Wrote gradcam CSVs + figures; checkpoint:", ckpts[0].name)
else:
    print("No proposed fold checkpoints found; run LOSO training first.")

### Generated artifacts

After a full run, CSVs live in `outputs/csv/` (see `README.md` table). Quick check:

In [ ]:
for p in sorted(cfg.csv_dir.glob("*.csv")):
    print(p.name)

### Manuscript-style tables (PRD Section 14)

Consolidated CSVs and `MANUSCRIPT_TABLES.md` under `outputs/reports/`, plus `outputs/logs/output_manifest.csv`.

In [ ]:
from stew_mwl.reports import build_manuscript_tables, write_run_manifest

table_paths = build_manuscript_tables(cfg)
write_run_manifest(cfg, extra={"config_path": str(cfg.config_path or "")})
print("Reports:", cfg.reports_dir)
for k, v in sorted(table_paths.items()):
    print(f"  {k}: {v.name}")